# खर्च दावा विश्लेषण

हा नोटबुक दाखवतो की स्थानिक पावती प्रतिमांकडून प्रवासाच्या खर्चाची प्रक्रिया करण्यासाठी प्लगइन्स वापरणारे एजंट कसे तयार करायचे, खर्च दावे ईमेल कसे तयार करायचे, आणि खर्च डेटा पाय चार्ट वापरून कसा दृश्यमान बनवायचा. एजंट कार्य संदर्भानुसार फंक्शन्स डायनॅमिकली निवडतात.

पद्धत:
1. OCR एजंट स्थानिक पावती प्रतिमा प्रक्रिया करतो आणि प्रवासाचा खर्च डेटा काढतो.
2. ईमेल एजंट खर्च दावा ईमेल तयार करतो.

### प्रवास खर्च परिस्थितीचे उदाहरण:
समजा तुम्ही दुसऱ्या शहरातील व्यावसायिक बैठकीसाठी प्रवास करणारा कर्मचारी आहात. तुमच्या कंपनीची धोरण सर्व योग्य प्रवास-संबंधित खर्च भरपाई करण्याची आहे. संभाव्य प्रवास खर्चांचे विभाजन पुढीलप्रमाणे:
- वाहतूक:
तुमच्या घरच्या शहरापासून गंतव्य शहरापर्यंत परतावा असलेल्या हवाई प्रवासासाठी विमान तिकीट.
हवामानशास्त्राला आणि हवाई अड्ड्याकडे टॅक्सी किंवा राईड-हेलिंग सेवा.
गंतव्य शहरातील स्थानिक वाहतूक (जसे की सार्वजनिक वाहतूक, भाड्याने देण्यात आलेली गाड्या किंवा टॅक्सी).

- निवास:
मीटिंगच्या ठिकाणाजवळील मध्यम श्रेणीतील व्यावसायिक हॉटेलमध्ये तीन रात्रींचा मुक्काम.

- जेवण:
कंपनीच्या प्रति दिवस धोरणावर आधारित नाश्ता, दुपारी जेवण आणि रात्री जेवणासाठी दररोजचे जेवण भत्ते.

- विविध खर्च:
हवाई अड्ड्यात पार्किंग शुल्क.
हॉटेलमध्ये इंटरनेट प्रवेश शुल्क.
टिप्स किंवा लहान सेवा शुल्क.

- दस्तऐवजीकरण:
तुम्ही सर्व पावत्या (फ्लाइट, टॅक्सी, हॉटेल, जेवण, इ.) आणि भरपाईसाठी पूर्णपणे भरलेला खर्च अहवाल जमा कराल.


## आवश्यक लायब्ररी आयात करा

नोटबुकसाठी आवश्यक लायब्ररी आणि मोड्यूल आयात करा.


In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import base64
import dotenv
from typing import Annotated, List

from pydantic import BaseModel, Field

from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Azure AI Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

 ## खर्च मॉडेल定義 करा

 वैयक्तिक खर्चांसाठी एक Pydantic मॉडेल तयार करा आणि वापरकर्ता क्वेरीला संरचित खर्च डेटा मध्ये रूपांतरित करण्यासाठी ExpenseFormatter वर्ग तयार करा.

 प्रत्येक खर्च खालील स्वरूपात दर्शविला जाईल:
 `{'date': '07-Mar-2025', 'description': 'flight to destination', 'amount': 675.99, 'category': 'Transportation'}`


In [ ]:
class Expense(BaseModel):
    date: str = Field(..., description="Date of expense in dd-MMM-yyyy format")
    description: str = Field(..., description="Expense description")
    amount: float = Field(..., description="Expense amount")
    category: str = Field(..., description="Expense category (e.g., Transportation, Meals, Accommodation, Miscellaneous)")

class ExpenseFormatter(BaseModel):
    raw_query: str = Field(..., description="Raw query input containing expense details")
    
    def parse_expenses(self) -> List[Expense]:
        """
        Parses the raw query into a list of Expense objects.
        Expected format: "date|description|amount|category" separated by semicolons.
        """
        expense_list = []
        for expense_str in self.raw_query.split(";"):
            if expense_str.strip():
                parts = expense_str.strip().split("|")
                if len(parts) == 4:
                    date, description, amount, category = parts
                    try:
                        expense = Expense(
                            date=date.strip(),
                            description=description.strip(),
                            amount=float(amount.strip()),
                            category=category.strip()
                        )
                        expense_list.append(expense)
                    except ValueError as e:
                        print(f"[LOG] Parse Error: Invalid data in '{expense_str}': {e}")
        return expense_list

## साधने परिभाषित करणे - ईमेल तयार करणे

खर्च दावा सादर करण्यासाठी ईमेल तयार करण्यासाठी एक साधन फंक्शन तयार करा.
- हे साधन Microsoft Agent Framework मधील `@tool` डेकॉरेटर वापरते.
- हे खर्चाची एकूण रक्कम गणना करते आणि तपशील ईमेल बॉडीमध्ये स्वरूपित करते.


In [ ]:
@tool(approval_mode="never_require")
def generate_expense_email(
    expense_data: Annotated[str, "Semicolon-separated expense entries in 'date|description|amount|category' format"]
) -> str:
    """Generate an email to submit an expense claim to the Finance Team."""
    formatter = ExpenseFormatter(raw_query=expense_data)
    expenses = formatter.parse_expenses()
    if not expenses:
        return "No valid expenses found to include in the email."
    total_amount = sum(e.amount for e in expenses)
    email_body = "Dear Finance Team,\n\n"
    email_body += "Please find below the details of my expense claim:\n\n"
    for e in expenses:
        email_body += f"- {e.date} | {e.description}: ${e.amount:.2f} ({e.category})\n"
    email_body += f"\nTotal Amount: ${total_amount:.2f}\n\n"
    email_body += "Receipts for all expenses are attached for your reference.\n\n"
    email_body += "Thank you,\n[Your Name]"
    return email_body

# पावती प्रतिमा मधून प्रवास खर्च काढण्याचे साधन

पावती प्रतिमा मधून प्रवास खर्च काढण्यासाठी एक साधन फंक्शन तयार करा.
- हे साधन Microsoft Agent Framework मधील `@tool` डेकॉरेटर वापरते.
- ते पावती प्रतिमा वाचते, त्याला base64 मध्ये एन्कोड करते, आणि एजंट साठी विश्लेषणासाठी डेटा URI परत करते.


In [ ]:
@tool(approval_mode="never_require")
def load_receipt_image(
    image_path: Annotated[str, "Path to the receipt image file"] = "receipt.jpg"
) -> str:
    """Load a receipt image and return its base64-encoded data URI for OCR extraction."""
    try:
        with open(image_path, "rb") as f:
            image_data = base64.b64encode(f.read()).decode("utf-8")
        return f"data:image/jpeg;base64,{image_data}"
    except Exception as e:
        error_msg = f"[LOG] Error loading image '{image_path}': {str(e)}"
        print(error_msg)
        return error_msg

## खर्च प्रक्रिया

एजंट्स निश्चित करा आणि त्यांना `WorkflowBuilder` वापरून एक अनुक्रमिक कार्यप्रवाहात जोडा.
- OCR एजंट `load_receipt_image` टूलचा वापर करून स्वीकारलेल्या पावतीच्या प्रतिमेमधून संरचित खर्च डेटा काढतो.
- ईमेल एजंट काढलेल्या डेटाचा वापर करून `generate_expense_email` टूलद्वारे व्यावसायिक खर्च दावा ईमेल तयार करतो.
- `WorkflowBuilder` मधील `add_edge` वापरून एक अनुक्रमिक पाइपलाइन तयार होते: OCR एजंट → ईमेल एजंट.


In [ ]:
ocr_agent = client.as_agent(
    tools=[load_receipt_image],
    name="OCRAgent",
    instructions=(
        "You are an expert OCR assistant specialized in extracting structured data from receipt images. "
        "Use the 'load_receipt_image' tool to load the receipt image, then analyze it and extract "
        "travel-related expense details in the format: 'date|description|amount|category' separated by semicolons. "
        "Follow these rules: "
        "- Date: Convert dates (e.g., '4/4/22') to 'dd-MMM-yyyy' (e.g., '04-Apr-2022'). "
        "- Description: Extract item names. "
        "- Amount: Use numeric values (e.g., '4.50' from '$4.50'). "
        "- Category: Infer from context (e.g., 'Meals' for food, 'Transportation' for travel, "
        "'Accommodation' for lodging, 'Miscellaneous' otherwise). "
        "Ignore totals, subtotals, or service charges unless they are itemized expenses. "
        "If no expenses are found, return 'No expenses detected'. "
        "Return only the structured data, no additional text."
    ),
)

email_agent = client.as_agent(
    name="EmailAgent",
    tools=[generate_expense_email],
    instructions=(
        "You are an expense claim email generator. Take the travel expense data from the previous agent "
        "(in 'date|description|amount|category' format separated by semicolons) and use the "
        "'generate_expense_email' tool to produce a professional expense claim email. "
        "Pass the semicolon-separated expense data directly to the tool."
    ),
)

## Main function

सिलसिलेवार वर्कफ्लो तयार करा आणि रसीद प्रतिमेवर प्रक्रिया करण्यासाठी आणि खर्च दावा ईमेल तयार करण्यासाठी तो चालवा.


> **टीप:** हा कार्यप्रवाह सध्या पावतीची प्रतिमा base64 मजकूर म्हणून पाठवतो, ज्याला बहुतेक चॅट मॉडेल्स (gpt-4o सह) प्रतिमा म्हणून विचारणार नाहीत.
> यामुळे मॉडेल संदर्भ विंडोही ओलांडू शकते. Azure AI Vision (किंवा इतर OCR साधन) सह OCR चालवणे आणि फक्त काढलेला मजकूर पाठवणे किंवा प्रतिमा `image_url` संदेश म्हणून पाठविण्यासाठी पुन्हा रचना करणे बरे.
> तुम्हाला फक्त संदर्भ त्रुटी टाळायच्या असतील तर, कमी मोठी पावती प्रतिमा वापरून पहा किंवा जास्त संदर्भ विंडो असलेले मॉडेल वापरा.


In [ ]:
workflow = WorkflowBuilder(start_executor=ocr_agent) \
    .add_edge(ocr_agent, email_agent) \
    .build()

prompt = (
    "Please extract the raw text from the receipt image at 'receipt.jpg', "
    "focusing on travel expenses like dates, descriptions, amounts, and categories "
    "(e.g., Transportation, Accommodation, Meals, Miscellaneous). "
    "Then generate a professional expense claim email."
)

last_author = None
events = workflow.run(
    prompt,
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"# Agent - {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**अस्वीकरण**:
हा दस्तऐवज AI भाषांतर सेवा [Co-op Translator](https://github.com/Azure/co-op-translator) चा वापर करून अनुवादित केला आहे. जरी आम्ही अचूकतेसाठी प्रयत्न करतो, तरी कृपया लक्षात घ्या की स्वयंचलित भाषांतरांमध्ये त्रुटी किंवा अचूकतेची कमतरता असू शकते. मूळ दस्तऐवज त्याच्या मूळ भाषेत अधिकृत स्रोत मानला पाहिजे. महत्त्वाची माहिती असल्यास, व्यावसायिक मानवी भाषांतराची शिफारस केली जाते. या भाषांतराच्या वापरामुळे उद्भवणाऱ्या कोणत्याही गैरसमज किंवा चुकीच्या अर्थलावणीसाठी आम्ही जबाबदार नाही.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
